# Preprocessing Pipeline

This notebook runs the full preprocessing pipeline for transforming raw DeepLabCut (DLC) tracking output into analysis-ready trial data. It mirrors the steps documented in `docs/Pipeline2024` and can serve as a replacement for that file.

**Pipeline overview:**

```
DLC output (.h5)
  |
  v
1. DLCFileSorting         Sort & stitch files by experimental condition
  |
  v
2. MappingRealWorld       2D DLC coordinates --> 3D real-world coordinates
  |
  v
3. GaitLabelling          Manual stance/swing labelling (GUI, run once)
4. GaitFeatureExtraction   Extract kinematic features from labelled frames
5. GaitClassification      Train LightGBM stance/swing classifier
  |
  v
6. GetRunsAndLoco         Segment trials, classify phases & limb states
  |
  v
7. CheckGoodRuns          (Optional) Validate run detection quality
  |
  v
8. FinalPrep              Compile per-condition pickle files
```

**Note:** Steps 3-5 (gait classifier training) only need to be run once unless the DLC tracking model changes significantly.

In [ ]:
import os
import sys

# Ensure project root is on the path so all package imports work
PROJECT_ROOT = os.path.dirname(os.path.abspath(""))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

---
## Step 1: DLC File Sorting

Sort raw DLC analysis files into folders organised by experimental condition and stitch any broken-up recordings into single files. Also handles one-off corrections (e.g. a mouse recorded separately that needs moving to the correct folder, and a mouse ID typo on one date).

| | |
|---|---|
| **Script** | `preprocessing/DLCFileSorting.py` |
| **Inputs** | Raw DLC `.h5` files in the DLC analysis output directories |
| **Outputs** | Organised files under `filtereddata_folder`, structured by condition (`exp_cats`) |

In [ ]:
from preprocessing.DLCFileSorting import copy_files_recursive, manual_changes, final_checks
from helpers.config import paths, exp_cats, micestuff

dlc_dest = paths['filtereddata_folder']
MouseIDs = micestuff['mice_IDs']
overwrite = False  # set True to overwrite existing files

copy_files_recursive(dlc_dest, exp_cats, '', MouseIDs, overwrite)
manual_changes()
final_checks()

---
## Step 2: 3D Coordinate Mapping

Map 2D DLC coordinates to 3D real-world space via camera calibration and triangulation. For each recording:
- Checks if the camera moved during the session
- Optimises the DLC-tracking-based camera calibration using reprojection error on labelled frames (via `helpers/OptimizeCalibration.py`)
- Saves `*_mapped3D.h5` files with 3D coordinates, plus calibration data and optimisation visualisations

Each condition is processed separately. Set `overwrite=True` to recompute existing files.

| | |
|---|---|
| **Script** | `preprocessing/MappingRealWorld.py` |
| **Inputs** | Sorted DLC `.h5` files, camera calibration data |
| **Outputs** | `*_mapped3D.h5` files with 3D coordinates |

In [ ]:
from preprocessing.MappingRealWorld import GetDirsFromConditions

# --- Repeats (3-day paradigm) ---
print("Mapping Repeats...")
GetDirsFromConditions(
    exp='APAChar', speed='LowHigh', repeat_extend='Repeats', exp_wash='Exp',
    overwrite=False
).get_dirs()

# --- Extended experiments ---
for speed in ['LowHigh', 'LowMid', 'HighLow']:
    print(f"Mapping Extended: {speed}...")
    GetDirsFromConditions(
        exp='APAChar', speed=speed, repeat_extend='Extended',
        overwrite=False
    ).get_dirs()

# --- Perception test ---
print("Mapping PerceptionTest...")
GetDirsFromConditions(exp='PerceptionTest', overwrite=False).get_dirs()

---
## Steps 3-5: Gait Classifier Training

Build a stance/swing classifier from manually labelled frames. These steps only need to be run once unless the DLC tracking model changes significantly.

### Step 3: Manual Gait Labelling (GUI)

Uses a specified selection of extracted frames from `labelling/MultiCamLabelling.py`. Label each frame as stance, swing, or unidentified for each limb using an interactive GUI. Recordings are selected to cover a range of mice, conditions, and dates.

| | |
|---|---|
| **Script** | `gait/GaitLabelling.py` |
| **Inputs** | Extracted frames from `labelling/MultiCamLabelling.py` |
| **Outputs** | `limb_labels.csv` |

In [ ]:
from gait.GaitLabelling import ImageLabeler

base_directory_side = r"H:\Dual-belt_APAs\analysis\DLC_DualBelt\Manual_Labelling\Side"
base_directory_front = r"H:\Dual-belt_APAs\analysis\DLC_DualBelt\Manual_Labelling\Front"
output_file = r"H:\Dual-belt_APAs\analysis\DLC_DualBelt\DualBelt_MyAnalysis\FilteredData\Round4_Oct24\LimbStuff\limb_labels.csv"

# Recordings selected to cover a range of mice, conditions, and dates
subdirectories_to_include = [
    "HM_20230306_APACharRepeat_FAA-1035243_None_side_1",
    "HM_20230306_APACharRepeat_FAA-1035245_R_side_1",
    "HM_20230309_APACharRepeat_FAA-1035297_R_side_1",
    "HM_20230306_APACharRepeat_FAA-1035244_L_side_1",
    "HM_20230307_APAChar_FAA-1035302_LR_side_1",
    "HM_20230308_APACharRepeat_FAA-1035244_L_side_1",
    "HM_20230319_APACharExt_FAA-1035245_R_side_1",
    "HM_20230326_APACharExt_FAA-1035246_LR_side_1",
    "HM_20230404_APACharExt_FAA-1035299_None_side_1",
    "HM_20230412_APACharExt_FAA-1035302_LR_side_1",
    "HM_20230309_APACharRepeat_FAA-1035301_R_side_1",
    "HM_20230325_APACharExt_FAA-1035249_R_side_1",
]

labeler = ImageLabeler(base_directory_side, base_directory_front, subdirectories_to_include, output_file)
if labeler.num_images == 0:
    print("No images found in the specified subdirectories.")
else:
    labeler.label_images()

### Step 4: Gait Feature Extraction

Extract kinematic features (positions, velocities, joint angles) from the 3D-mapped data at each labelled frame. Features are computed at 7 time offsets (-10, -5, -1, 0, +1, +5, +10 frames) to capture temporal context.

| | |
|---|---|
| **Script** | `gait/GaitFeatureExtraction.py` |
| **Inputs** | `limb_labels.csv`, 3D-mapped coordinate files |
| **Outputs** | `extracted_features.csv` |

In [ ]:
from gait.GaitFeatureExtraction import RunClassifier

base_directory = r"H:\Dual-belt_APAs\analysis\DLC_DualBelt\DualBelt_MyAnalysis\FilteredData\Round4_Oct24\LimbStuff"

classifier = RunClassifier(base_directory)
classifier.collect_features()
classifier.save_features(os.path.join(base_directory, 'extracted_features.csv'))

### Step 5: Gait Classification

Train a multi-output LightGBM classifier to predict stance/swing state for each of the 4 limbs. Outputs confusion matrices and feature importance plots for evaluation.

| | |
|---|---|
| **Script** | `gait/GaitClassification.py` |
| **Inputs** | `extracted_features.csv` |
| **Outputs** | Trained model (`.pkl`), label encoders, confusion matrices, feature importance plots |

In [ ]:
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from gait.GaitClassification import (
    load_and_preprocess_data, train_model,
    multiclass_multioutput_hamming_loss, analyze_misclassifications, plot_feature_importance,
)

data_path = r"H:\Dual-belt_APAs\analysis\DLC_DualBelt\DualBelt_MyAnalysis\FilteredData\Round4_Oct24\LimbStuff\extracted_features.csv"
output_dir = r"H:\Dual-belt_APAs\analysis\DLC_DualBelt\DualBelt_MyAnalysis\FilteredData\Round4_Oct24\LimbStuff"
label_columns = ['ForepawR', 'ForepawL', 'HindpawR', 'HindpawL']

# --- Load & preprocess ---
X, y, feature_columns, label_encoders = load_and_preprocess_data(data_path, label_columns)

# Save label encoders and feature columns for use in later steps
joblib.dump(label_encoders, os.path.join(output_dir, 'label_encoders.pkl'))
joblib.dump(feature_columns, os.path.join(output_dir, 'feature_columns.pkl'))

# --- Train/test split ---
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.reset_index(drop=True, inplace=True)
X_test.reset_index(drop=True, inplace=True)
y_train.reset_index(drop=True, inplace=True)
y_test.reset_index(drop=True, inplace=True)

X_train_features = X_train[feature_columns]
X_test_features = X_test[feature_columns]

# --- Train ---
multi_target_model = train_model(X_train_features, y_train, output_dir, model_type='lightgbm')

# --- Evaluate ---
y_pred = multi_target_model.predict(X_test_features)

for idx, col in enumerate(label_columns):
    accuracy = accuracy_score(y_test[col], y_pred[:, idx])
    print(f"\n{'='*50}")
    print(f"{col}  |  Accuracy: {accuracy:.4f}")
    print(classification_report(y_test[col], y_pred[:, idx], target_names=label_encoders[col].classes_))

    cm = confusion_matrix(y_test[col], y_pred[:, idx])
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_encoders[col].classes_).plot()
    plt.title(f'Confusion Matrix: {col}')
    plt.savefig(os.path.join(output_dir, f'confusion_matrix_{col}.png'))
    plt.show()

hloss = multiclass_multioutput_hamming_loss(y_test, y_pred)
subset_accuracy = (y_pred == y_test.values).all(axis=1).mean()
print(f"\nOverall Hamming Loss:   {hloss:.4f}")
print(f"Exact Match Accuracy:   {subset_accuracy:.4f}")

# --- Diagnostics ---
base_directory_side = r"H:\Dual-belt_APAs\analysis\DLC_DualBelt\Manual_Labelling\Side"
analyze_misclassifications(X_test, y_test, y_pred, label_columns, base_directory_side, output_dir, label_encoders)
plot_feature_importance(multi_target_model, feature_columns, output_dir)
plt.show()

---
## Step 6: Trial Segmentation & Limb State Classification

Segment continuous recordings into individual trials (door open &rarr; door close), classify each frame into run phases (`TrialStart`, `RunStart`, `Transition`, `RunEnd`, `TrialEnd`, `RunBack`), and apply the trained gait classifier to label each limb as stance or swing.

| | |
|---|---|
| **Script** | `preprocessing/GetRunsAndLoco.py` |
| **Inputs** | 3D-mapped coordinate files, trained gait classifier |
| **Outputs** | Per-recording trial data with phase and limb-state labels |

In [ ]:
from preprocessing.GetRunsAndLoco import GetConditionFiles

# --- Repeats (covers Days 1-3) ---
print("Processing Repeats...")
GetConditionFiles(
    exp='APAChar', speed='LowHigh', repeat_extend='Repeats', exp_wash='Exp',
    overwrite=False, save_frames=True
).get_dirs()

# --- Extended experiments ---
for speed in ['LowHigh', 'LowMid', 'HighLow']:
    print(f"Processing Extended: {speed}...")
    GetConditionFiles(
        exp='APAChar', speed=speed, repeat_extend='Extended',
        overwrite=False, save_frames=True
    ).get_dirs()

---
## Step 7 (Optional): Validate Run Detection

Plot run availability across experiments to visualise any missing data or detection failures. Useful as a sanity check before final compilation.

| |                                         |
|---|-----------------------------------------|
| **Script** | `preprocessing/CheckGoodRuns.py` |
| **Inputs** | Trial-segmented data from Step 6        |
| **Outputs** | Run availability plots                  |

In [ ]:
from preprocessing.CheckGoodRuns import GetConditionFiles as CheckConditionFiles

# --- Repeats ---
CheckConditionFiles(
    exp='APAChar', speed='LowHigh', repeat_extend='Repeats', exp_wash='Exp',
    overwrite=False
).get_dirs()

# --- Extended ---
for speed in ['LowHigh', 'LowMid', 'HighLow']:
    CheckConditionFiles(
        exp='APAChar', speed=speed, repeat_extend='Extended',
        overwrite=False
    ).get_dirs()

---
## Step 8: Final Data Compilation

Compile all per-mouse files within each condition into single pickle files. Each file contains a dictionary of DataFrames keyed by mouse ID. A discrete swing/stance column (`SwSt_discrete`) is added. Extended experiment days are also concatenated into unified files.

| | |
|---|---|
| **Script** | `preprocessing/FinalPrep.py` |
| **Inputs** | Trial-segmented data from Step 6 |
| **Outputs** | Per-condition `.pkl` files ready for analysis |

In [ ]:
from preprocessing.FinalPrep import GetConditionFiles as FinalPrepFiles

# --- Repeats (per-day) ---
for day in ['Day1', 'Day2', 'Day3']:
    print(f"Compiling Repeats {day}...")
    FinalPrepFiles(
        exp='APAChar', speed='LowHigh', repeat_extend='Repeats', exp_wash='Exp', day=day
    ).get_dirs()

# --- Extended ---
for speed in ['LowHigh', 'LowMid', 'HighLow']:
    print(f"Compiling Extended: {speed}...")
    FinalPrepFiles(
        exp='APAChar', speed=speed, repeat_extend='Extended'
    ).get_dirs()